# Canonical WOMD Paper Pipeline (5 seeds / 20 checkpoints)

Production research pipeline for Stages 1–7. Four objectives × five seeds = 20 checkpoints. Lambda selection is development-only; official validation remains untouched until Stage 5.


In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
ROOT = Path('/content/predictive-pc-fmcw')
DATA = Path('/content/womd')
OUT = Path('/content/drive/MyDrive/predictive_pc_fmcw_canonical')
DATA.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
REPO = 'https://github.com/panagiotagrosdouli/predictive-pc-fmcw-vehicular-communications..git'
if not ROOT.exists(): subprocess.run(['git','clone','--depth','1',REPO,str(ROOT)], check=True)
else: subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[ml,paper]','--no-build-isolation'], check=True)


## Stage 1 — deterministic WOMD recovery and provenance gates
Historical fingerprint is evidence, not a preprocessing target. A mismatch is recorded for scientific explanation; corpus integrity and train/validation isolation remain fail-closed.


In [ ]:
BUCKET='gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario'
train_dir=DATA/'training'; val_dir=DATA/'validation'; train_dir.mkdir(exist_ok=True); val_dir.mkdir(exist_ok=True)
def fetch_split(split,count,total,dst):
    for i in range(count):
        name=f'{split}.tfrecord-{i:05d}-of-{total:05d}'; target=dst/name
        if not target.exists(): subprocess.run(['gcloud','storage','cp',f'{BUCKET}/{split}/{name}',str(target)],check=True)
fetch_split('training',50,1000,train_dir); fetch_split('validation',40,150,val_dir)
train_npz=DATA/'womd_training_paper.npz'; val_npz=DATA/'womd_validation_paper.npz'
subprocess.run([sys.executable,str(ROOT/'scripts/01_build_official_womd_samples.py'),*map(str,sorted(train_dir.glob('training.tfrecord-*'))),'--output',str(train_npz),'--max-vehicles','16'],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/01_build_official_womd_samples.py'),*map(str,sorted(val_dir.glob('validation.tfrecord-*'))),'--output',str(val_npz),'--max-vehicles','16','--fixed-split','official_validation'],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/08_audit_womd_dataset.py'),str(train_npz),'--output',str(OUT/'training_audit.json')],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/08_audit_womd_dataset.py'),str(val_npz),'--output',str(OUT/'validation_audit.json')],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/01_verify_historical_womd_fingerprint.py'),str(train_npz),'--output',str(OUT/'historical_fingerprint.json')],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/01_verify_womd_corpora.py'),str(train_npz),str(val_npz),'--output',str(OUT/'corpus_verification.json')],cwd=ROOT,check=True)
print('Stage 1 integrity gates PASS; inspect historical_fingerprint.json before declaring exact reproduction.')


## Stages 3–4 — development baselines, lambda freeze, canonical training


In [ ]:
SEEDS=['20260827','20260828','20260829','20260830','20260831']
BER=ROOT/'artifacts/paper_final/02_link/dbpsk_ber_lut.csv'
assert BER.is_file(), 'Stage 2 BER LUT missing; complete Stage 2 before learned experiments.'
subprocess.run([sys.executable,str(ROOT/'scripts/03_eval_npz_baselines.py'),str(train_npz),'--split','development','--ber-lut',str(BER),'--output',str(ROOT/'artifacts/paper_final/03_baselines')],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/04_run_lambda_sweep.py'),str(train_npz),'--ber-lut',str(BER),'--output',str(ROOT/'artifacts/paper_final/04_learning/lambda_sweep'),'--epochs','80','--batch-size','32','--seeds',*SEEDS],cwd=ROOT,check=True)


In [ ]:
# Freeze only after inspecting development-only sweep. Never tune on official validation.
LAMBDA_LINK='0.2'; LAMBDA_OUTAGE='0.1'
RATIONALE='Selected from declared development-only sweep; frozen before Stage 5.'
selection=ROOT/'artifacts/paper_final/04_learning/lambda_selection.json'
subprocess.run([sys.executable,str(ROOT/'scripts/04_freeze_lambda_selection.py'),str(ROOT/'artifacts/paper_final/04_learning/lambda_sweep'),str(train_npz),'--lambda-link',LAMBDA_LINK,'--lambda-outage',LAMBDA_OUTAGE,'--rationale',RATIONALE,'--output',str(selection)],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/04_run_training_ablation.py'),str(train_npz),'--ber-lut',str(BER),'--output',str(ROOT/'artifacts/paper_final/04_learning/learned_ablation'),'--epochs','80','--batch-size','32','--seeds',*SEEDS,'--selection',str(selection)],cwd=ROOT,check=True)
checkpoints=sorted((ROOT/'artifacts/paper_final/04_learning/learned_ablation').rglob('best_comm_aware_gru.pt'))
assert len(checkpoints)==20, f'Canonical archive incomplete: {len(checkpoints)}/20 checkpoints'


## Stages 5–7 — untouched validation, paired scheduling, scenario-level statistics


In [ ]:
completion=ROOT/'artifacts/paper_final/04_learning/learned_ablation/completion_manifest.json'
assert completion.is_file(), 'Canonical completion manifest missing.'
heldout=ROOT/'artifacts/paper_final/05_heldout'
subprocess.run([sys.executable,str(ROOT/'scripts/06_evaluate_learned_checkpoints.py'),str(val_npz),*map(str,checkpoints),'--development-npz',str(train_npz),'--completion-manifest',str(completion),'--ber-lut',str(BER),'--output',str(heldout)],cwd=ROOT,check=True)
sched=ROOT/'artifacts/paper_final/06_scheduling'
subprocess.run([sys.executable,str(ROOT/'scripts/06_evaluate_learned_scheduler_womd.py'),*map(str,sorted(val_dir.glob('validation.tfrecord-*'))),'--checkpoints',*map(str,checkpoints),'--training-npz',str(train_npz),'--completion-manifest',str(completion),'--ber-lut',str(BER),'--heldout-metrics',str(heldout/'heldout_metrics_by_scenario.csv'),'--output',str(sched)],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/07_analyze_learned_results.py'),str(heldout/'heldout_metrics_by_scenario.csv'),'--output',str(ROOT/'artifacts/paper_final/07_analysis/learned')],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/07_analyze_scheduler_utility.py'),str(heldout/'heldout_metrics_by_scenario.csv'),str(sched),'--output',str(ROOT/'artifacts/paper_final/07_analysis/statistics')],cwd=ROOT,check=True)
